# 🔍 Investigate HDF5 Data Files

**Explore the structure and contents of vifactcheck_*.h5 files**

📂 Files: `../processed_data/hdf5/vifactcheck_{train,dev,test}.h5`

**This notebook will:**
- Display HDF5 file structure
- Show dataset shapes and dtypes
- Visualize sample data
- Implement PyTorch DataLoader
- Compare train/dev/test splits

In [ ]:
# Install dependencies
import subprocess
import sys

packages = ["h5py", "torch", "numpy", "matplotlib", "tqdm"]

for pkg in packages:
    try:
        __import__(pkg)
        print(f"✅ {pkg}")
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        print(f"📦 Installed {pkg}")

In [ ]:
# Setup
import sys
import json
import h5py
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

# Paths
HDF5_DIR = project_root / "processed_data" / "hdf5"
SPLITS = ["train", "dev", "test"]

print(f"✅ Project root: {project_root}")
print(f"📂 HDF5 directory: {HDF5_DIR}")

In [ ]:
# Function to investigate HDF5 structure
def investigate_hdf5(filepath):
    """Display structure of HDF5 file."""
    print(f"\n{'='*60}")
    print(f"📂 File: {filepath.name}")
    print(f"{'='*60}")
    
    with h5py.File(filepath, "r") as f:
        print(f"\n🗂️  Datasets:")
        for key in f.keys():
            ds = f[key]
            compression = ds.compression if ds.compression else "none"
            size_mb = ds.nbytes / 1024 / 1024
            print(f"  • {key}")
            print(f"    Shape: {ds.shape}, Dtype: {ds.dtype}")
            print(f"    Compression: {compression}, Size: {size_mb:.1f} MB")
    
    return filepath

# Investigate all splits
hdf5_files = {}
for split in SPLITS:
    filepath = HDF5_DIR / f"vifactcheck_{split}.h5"
    if filepath.exists():
        investigate_hdf5(filepath)
        hdf5_files[split] = filepath
    else:
        print(f"⚠️  File not found: {filepath}")

In [ ]:
# Load sample data from each split
def show_samples(split, filepath, num_samples=3):
    """Display sample data from HDF5."""
    print(f"\n{'='*60}")
    print(f"📊 {split.upper()} Samples")
    print(f"{'='*60}")
    
    with h5py.File(filepath, "r") as f:
        # Get shapes
        n_samples = f["labels"].shape[0]
        print(f"Total samples: {n_samples}")
        
        # Load sample indices
        indices = [0, n_samples//2, n_samples-1]
        
        for idx in indices[:num_samples]:
            print(f"\n--- Sample {idx} ---")
            print(f"Label: {f['labels'][idx]}")
            
            # Caption
            caption = f["captions"][idx]
            if isinstance(caption, bytes):
                caption = caption.decode("utf-8")
            print(f"Caption: {caption[:100]}...")
            
            # Text features
            text_feat = f["text_features"][idx]
            print(f"Text features: {text_feat.shape}, mean={text_feat.mean():.4f}, std={text_feat.std():.4f}")
            
            # Image features
            img_feat = f["image_features"][idx]
            print(f"Image features: {img_feat.shape}, mean={img_feat.mean():.4f}, std={img_feat.std():.4f}")

for split, filepath in hdf5_files.items():
    show_samples(split, filepath)

In [ ]:
# Compare splits
def compare_splits(hdf5_files):
    """Compare train/dev/test splits."""
    stats = {}
    
    for split, filepath in hdf5_files.items():
        with h5py.File(filepath, "r") as f:
            labels = f["labels"][:]
            stats[split] = {
                "n_samples": len(labels),
                "n_fake": int(labels.sum()),
                "n_real": int(len(labels) - labels.sum()),
                "fake_ratio": float(labels.mean()),
            }
    
    print(f"\n{'='*60}")
    print("📈 Split Comparison")
    print(f"{'='*60}\n")
    
    for split, s in stats.items():
        print(f"{split.upper()}:")
        print(f"  Samples: {s['n_samples']}")
        print(f"  Fake: {s['n_fake']} ({s['fake_ratio']*100:.1f}%)")
        print(f"  Real: {s['n_real']} ({(1-s['fake_ratio'])*100:.1f}%)")
        print()
    
    return stats

stats = compare_splits(hdf5_files)

In [ ]:
# Visualize label distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (split, filepath) in enumerate(hdf5_files.items()):
    with h5py.File(filepath, "r") as f:
        labels = f["labels"][:]
        
    axes[idx].bar(["Real", "Fake"], [len(labels)-labels.sum(), labels.sum()])
    axes[idx].set_title(f"{split.upper()} (n={len(labels)})")
    axes[idx].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# PyTorch Dataset class for HDF5
class HDF5Dataset(Dataset):
    """PyTorch Dataset for loading HDF5 multimodal data."""
    
    def __init__(self, hdf5_path):
        self.hdf5_path = hdf5_path
        
        # Get dataset size
        with h5py.File(hdf5_path, "r") as f:
            self.n_samples = f["labels"].shape[0]
            print(f"📊 Loaded dataset: {self.n_samples} samples")
    
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        with h5py.File(self.hdf5_path, "r") as f:
            text_features = torch.from_numpy(f["text_features"][idx]).float()
            image_features = torch.from_numpy(f["image_features"][idx]).float()
            label = torch.tensor(f["labels"][idx]).long()
            
            caption = f["captions"][idx]
            if isinstance(caption, bytes):
                caption = caption.decode("utf-8")
            
        return {
            "text_features": text_features,
            "image_features": image_features,
            "label": label,
            "caption": caption,
            "idx": idx,
        }

print("✅ HDF5Dataset class defined")

In [ ]:
# Create DataLoaders for all splits
BATCH_SIZE = 32

dataloaders = {}
datasets = {}

for split in SPLITS:
    filepath = HDF5_DIR / f"vifactcheck_{split}.h5"
    if filepath.exists():
        print(f"\n🔧 Creating {split} DataLoader...")
        
        # Create dataset
        dataset = HDF5Dataset(filepath)
        datasets[split] = dataset
        
        # Create dataloader
        shuffle = (split == "train")
        dataloader = DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=shuffle,
            num_workers=0,  # Use 0 for HDF5
        )
        dataloaders[split] = dataloader
        
        print(f"✅ {split}: {len(dataloader)} batches")

print(f"\n{'='*60}")
print("✅ All DataLoaders ready!")

In [ ]:
# Test DataLoader iteration
print("🔄 Testing DataLoader batch iteration...\n")

for split, dataloader in dataloaders.items():
    print(f"\n{'='*60}")
    print(f"📦 {split.upper()} DataLoader")
    print(f"{'='*60}")
    
    # Get first batch
    batch = next(iter(dataloader))
    
    print(f"Batch keys: {batch.keys()}")
    print(f"Text features shape: {batch['text_features'].shape}")
    print(f"Image features shape: {batch['image_features'].shape}")
    print(f"Labels shape: {batch['label'].shape}")
    print(f"Captions (first 2): {batch['caption'][:2]}")
    print(f"Indices: {batch['idx'][:5]}")
    
    # Verify data types
    print(f"\nData types:")
    print(f"  text_features: {batch['text_features'].dtype}")
    print(f"  image_features: {batch['image_features'].dtype}")
    print(f"  label: {batch['label'].dtype}")
    break  # Just show first split

In [ ]:
# Summary
print(f"{'='*60}")
print("📋 INVESTIGATION SUMMARY")
print(f"{'='*60}\n")

for split, filepath in hdf5_files.items():
    with h5py.File(filepath, "r") as f:
        n = f["labels"].shape[0]
        text_shape = f["text_features"].shape
        img_shape = f["image_features"].shape
        size_mb = filepath.stat().st_size / 1024 / 1024
        
    print(f"{split.upper()}:")
    print(f"  File: {filepath.name}")
    print(f"  Size: {size_mb:.1f} MB")
    print(f"  Samples: {n}")
    print(f"  Text features: {text_shape}")
    print(f"  Image features: {img_shape}")
    print()

print(f"{'='*60}")
print("✅ HDF5 files ready for training!")
print(f"{'='*60}")
print("\nUsage:")
print("  train_loader = dataloaders['train']")
print("  dev_loader = dataloaders['dev']")
print("  test_loader = dataloaders['test']")